In [1]:
from pathlib import Path

project_dir = Path.cwd()
raw_dir = project_dir / "data" / "raw"

print("Project directory:", project_dir)
print("Raw data directory:", raw_dir)

pdf_files = list(raw_dir.glob("*.pdf"))
print("Number of PDF documents:", len(pdf_files))

for file in pdf_files:
    print("-", file.name)


Project directory: C:\Users\Helen\Documents\mamacare-ai-danger-sign-assistant
Raw data directory: C:\Users\Helen\Documents\mamacare-ai-danger-sign-assistant\data\raw
Number of PDF documents: 3
- Nigeria_MPCDSR_2022.pdf
- WHO_Antenatal_Care_2016.pdf
- WHO_Essential_Practice_2023.pdf


import fitz  # PyMuPDF
from pathlib import Path

raw_dir = Path.cwd() / "data" / "raw"

print("Knowledge Base Document Summary\n")

for pdf_file in raw_dir.glob("*.pdf"):
    doc = fitz.open(pdf_file)
    num_pages = len(doc)

    first_page = doc.load_page(0).get_text("text")
    first_page = first_page.replace("\n", " ")[:150]

    print(f"Document: {pdf_file.name}")
    print(f"Pages: {num_pages}")
    print(f"First page preview: {first_page}")
    print("-" * 60)

    doc.close()

In [ ]:
## Initial Exploratory Data Analysis (EDA)

The knowledge base contains three authoritative maternal-health documents from the World Health Organization (WHO) and the Federal Ministry of Health and Social Welfare, Nigeria. The documents were successfully detected and opened using Python and PyMuPDF, confirming that they are available for preprocessing and retrieval.

| Document                        | Pages | Relevance to Maternal Danger-Sign Education |
| ------------------------------- | ----: | ------------------------------------------- |
| Nigeria_MPCDSR_2022.pdf         |   144 | High                                        |
| WHO_Antenatal_Care_2016.pdf     |   172 | High                                        |
| WHO_Essential_Practice_2023.pdf |   184 | Very High                                   |

The combined knowledge base contains approximately **500 pages** of maternal-health guidance. The documents include information on pregnancy complications, maternal danger signs, emergency referral, and safe maternal care practices. These findings confirm that the dataset is suitable for text extraction, document chunking, metadata tagging, and Retrieval-Augmented Generation (RAG) development.


import fitz
from pathlib import Path

raw_dir = Path.cwd() / "data" / "raw"

danger_keywords = [
    "bleeding",
    "headache",
    "blurred vision",
    "convulsion",
    "seizure",
    "abdominal pain",
    "difficulty breathing",
    "fever",
    "swelling",
    "fetal movement"
]

print("Danger-sign keyword search\\n")

for pdf_file in raw_dir.glob("*.pdf"):
    doc = fitz.open(pdf_file)
    full_text = ""

    for page in doc:
        full_text += page.get_text("text").lower()

    print(f"Document: {pdf_file.name}")

    for keyword in danger_keywords:
        count = full_text.count(keyword)
        print(f"  {keyword}: {count}")

    print("-" * 50)

    doc.close()

In [ ]:
### Keyword Analysis Findings

A keyword search was performed across the three knowledge-base documents using common maternal danger-sign terms. The results show that **WHO_Essential_Practice_2023.pdf** contains the highest frequency of clinically important danger-sign terms, including bleeding, abdominal pain, convulsions, headache, blurred vision, swelling, fever, and difficulty breathing. **WHO_Antenatal_Care_2016.pdf** contains additional antenatal warning-sign content, particularly fetal movement and abdominal pain. **Nigeria_MPCDSR_2022.pdf** provides Nigerian-specific maternal complication and referral guidance.

These findings indicate that the selected documents are appropriate for developing a bilingual maternal danger-sign education and referral assistant. The WHO Essential Practice document will be used as the primary knowledge source for retrieval, while the WHO Antenatal Care and Nigeria MPCDSR documents will provide supporting evidence and local clinical context.


## Data Dictionary

The data dictionary defines the structure of the knowledge-base records that will be created after text extraction and document chunking.

| Field            | Description                               | Example                                                       |
| ---------------- | ----------------------------------------- | ------------------------------------------------------------- |
| Document_ID      | Unique identifier for the source document | WHO2016                                                       |
| Source           | Organization that published the document  | WHO                                                           |
| Title            | Title of the document                     | WHO Recommendations on Antenatal Care                         |
| Publication_Year | Year the document was published           | 2016                                                          |
| Section_Title    | Name of the chapter or subsection         | Danger Signs in Pregnancy                                     |
| Page_Number      | Original page number in the PDF           | 45                                                            |
| Topic            | Main subject of the text chunk            | Vaginal Bleeding                                              |
| Pregnancy_Stage  | Pregnancy period covered                  | Third Trimester                                               |
| Language         | Language of the stored text               | English                                                       |
| Chunk_Text       | Extracted passage used for retrieval      | Women with vaginal bleeding should seek immediate assessment. |
| Version          | Document edition or version               | 2016 Edition                                                  |


### Why a Data Dictionary is Needed

The RAG system will not search entire PDF documents directly. Instead, each document will be divided into small searchable sections called **chunks**. Every chunk will have metadata such as the source, topic, pregnancy stage, page number, and document version. This metadata allows the retrieval system to find the most relevant evidence when a pregnant woman asks a question such as “I am bleeding” or “I get serious headache.”

Using a structured data dictionary improves retrieval accuracy, supports traceability of evidence, and makes it possible to cite the original WHO or Nigerian guideline used in each response.


In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
print(pd.__version__)

2.3.3


In [9]:
import pandas as pd

sample_chunks = pd.DataFrame([
    {
        "Document_ID": "WHO2016",
        "Source": "WHO",
        "Publication_Year": 2016,
        "Topic": "Fetal Movement",
        "Pregnancy_Stage": "Third Trimester",
        "Language": "English",
        "Chunk_Text": "Women should seek medical assessment if fetal movements decrease significantly."
    },
    {
        "Document_ID": "WHOEP",
        "Source": "WHO",
        "Publication_Year": 2023,
        "Topic": "Vaginal Bleeding",
        "Pregnancy_Stage": "Any Stage",
        "Language": "English",
        "Chunk_Text": "Vaginal bleeding during pregnancy requires urgent medical assessment."
    },
    {
        "Document_ID": "MPCDSR2022",
        "Source": "Federal Ministry of Health, Nigeria",
        "Publication_Year": 2022,
        "Topic": "Referral",
        "Pregnancy_Stage": "Any Stage",
        "Language": "English",
        "Chunk_Text": "Women with severe danger signs should be referred immediately to a higher-level facility."
    }
])

sample_chunks

,Document_ID,Source,Publication_Year,Topic,Pregnancy_Stage,Language,Chunk_Text
0,WHO2016,WHO,2016,Fetal Movement,Third Trimester,English,Women should seek medical assessment if fetal ...
1,WHOEP,WHO,2023,Vaginal Bleeding,Any Stage,English,Vaginal bleeding during pregnancy requires urg...
2,MPCDSR2022,"Federal Ministry of Health, Nigeria",2022,Referral,Any Stage,English,Women with severe danger signs should be refer...


In [10]:
import fitz
from pathlib import Path

raw_dir = Path.cwd() / "data" / "raw"

pdf_file = raw_dir / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

print("Document:", pdf_file.name)
print("Pages:", len(doc))
print()

# Extract text from page 20 (change the number if needed)
page = doc.load_page(20)
text = page.get_text("text")

print(text[:2000])  # show the first 2000 characters

doc.close()

Document: WHO_Essential_Practice_2023.pdf
Pages: 184




In [11]:
import fitz
from pathlib import Path

pdf_file = Path.cwd() / "data" / "raw" / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

# Check the first 10 pages for readable text
for i in range(10):
    text = doc.load_page(i).get_text("text")
    print(f"Page {i+1}: {len(text)} characters")

doc.close()

Page 1: 12 characters
Page 2: 147 characters
Page 3: 2232 characters
Page 4: 1719 characters
Page 5: 2815 characters
Page 6: 2194 characters
Page 7: 2661 characters
Page 8: 2027 characters
Page 9: 2817 characters
Page 10: 2776 characters


### Text Extraction Assessment

A text extraction assessment was performed on the WHO Essential Practice document using PyMuPDF. The results showed that Pages 3–10 contain between approximately **1,700 and 2,800 characters of readable text**, indicating that the document is suitable for automated text extraction and Retrieval-Augmented Generation (RAG) processing.

This confirms that the knowledge base documents can be converted into machine-readable text, divided into searchable chunks, and indexed for evidence-based retrieval.


In [12]:
import fitz
from pathlib import Path

pdf_file = Path.cwd() / "data" / "raw" / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

page = doc.load_page(4)   # Page 5 in the PDF
text = page.get_text("text")

print(text)

doc.close()

ACKNOWLEDGEMENTS
The 2006 edition was prepared by a team of the World Health Organization, Department of 
Reproductive Health and Research (RHR), led by Jerker Liljestrand and Jelka Zupan.
The concept and first drafts were developed by Sandra Gove and Patricia Whitesell/ACT International, 
Atlanta, Jerker Liljestrand, Denise Roth, Betty Sweet, Anne Thompson, and Jelka Zupan.
Revisions were subsequently carried out by Annie Portela, Luc de Bernis, Ornella Lincetto, Rita Kabra, 
Maggie Usher, Agostino Borra, Rick Guidotti, Elisabeth Hoff, Mathews Matthai, Monir Islam, 
Felicity Savage, Adepeyu Olukoya, Aafje Rietveld, TinTin Sint, Ekpini Ehounou, Suman Mehta.
Valuable inputs were provided by WHO Regional Offices and WHO departments:

 Reproductive Health and Research (RHR)

 Maternal, Newborn, Child and Adolescent Health (MCA)

 HIV/AIDS

 Nutrition for Health and Development (NHD)

 Essential Medicines and Health Products (EMP)

 Immunization, Vaccines and Biologicals (IVB)


In [13]:
import fitz
from pathlib import Path

pdf_file = Path.cwd() / "data" / "raw" / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

for page_num in range(len(doc)):
    text = doc.load_page(page_num).get_text("text").lower()
    if "bleeding" in text:
        print(f"The first page containing 'bleeding' is page {page_num + 1}")
        break

doc.close()

The first page containing 'bleeding' is page 6


In [14]:
import fitz
from pathlib import Path

pdf_file = Path.cwd() / "data" / "raw" / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

# Page 6 in the PDF = index 5
page = doc.load_page(5)
text = page.get_text("text")

print(text)

doc.close()

TABLE OF CONTENTS
	
INTRODUCTION
i   Introduction
i2   How to read the Guide
i3   Structure and presentation
i4   Assumptions underlying the guide
A PRINCIPLES OF GOOD CARE
A2   Communication
A3   Workplace and administrative procedures
A4   Standard precautions and cleanliness
A5   Organising a visit
B QUICK CHECK, RAPID ASSESSMENT AND MANAGEMENT OF WOMEN OF CHILDBEARING AGE
B2   Quick check
B3-B7	  Rapid assessment and management
	
B3 	 Airway and breathing
	
B3 	 Circulation (shock)
	
B4-B5	 Vaginal bleeding
	
B6 	 Convulsions or unconscious
	
B6 	 Severe abdominal pain
	
B6 	 Dangerous fever
	
B7 	 Labour
	
B7 	 Other danger signs or symptoms
	
B7 	 If no emergency or priority signs, non urgent
B EMERGENCY TREATMENTS FOR THE WOMAN
B9   Airway, breathing and circulation
	
B9 	 Manage the airway and breathing
	
B9 	 Insert IV line and give fluids
	
B9 	 If intravenous access not possible
	 B10-B12	 Bleeding
	
B10	 Massage uterus and expel clots
	
B10	 Apply bimanual uterine compressi

### Table of Contents Analysis

The WHO Essential Practice document contains dedicated clinical sections on **vaginal bleeding, convulsions or unconsciousness, severe abdominal pain, dangerous fever, pre-eclampsia/eclampsia, and urgent referral to hospital**. These topics correspond directly to the maternal danger signs that the MamaCare AI assistant is designed to detect and explain.

This confirms that the WHO Essential Practice document is the **primary knowledge source** for the MVP. The document provides both educational information and emergency referral guidance, making it suitable for retrieval-augmented generation (RAG) and rule-based danger-sign detection.


In [15]:
import fitz
from pathlib import Path

pdf_file = Path.cwd() / "data" / "raw" / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

# Search for the phrase "Vaginal bleeding"
for page_num in range(len(doc)):
    text = doc.load_page(page_num).get_text("text")
    if "Vaginal bleeding" in text:
        print(f"Found on page {page_num + 1}")
        print("-" * 50)
        print(text[:2500])   # print first 2500 characters
        break

doc.close()

Found on page 6
--------------------------------------------------
TABLE OF CONTENTS
	
INTRODUCTION
i   Introduction
i2   How to read the Guide
i3   Structure and presentation
i4   Assumptions underlying the guide
A PRINCIPLES OF GOOD CARE
A2   Communication
A3   Workplace and administrative procedures
A4   Standard precautions and cleanliness
A5   Organising a visit
B QUICK CHECK, RAPID ASSESSMENT AND MANAGEMENT OF WOMEN OF CHILDBEARING AGE
B2   Quick check
B3-B7	  Rapid assessment and management
	
B3 	 Airway and breathing
	
B3 	 Circulation (shock)
	
B4-B5	 Vaginal bleeding
	
B6 	 Convulsions or unconscious
	
B6 	 Severe abdominal pain
	
B6 	 Dangerous fever
	
B7 	 Labour
	
B7 	 Other danger signs or symptoms
	
B7 	 If no emergency or priority signs, non urgent
B EMERGENCY TREATMENTS FOR THE WOMAN
B9   Airway, breathing and circulation
	
B9 	 Manage the airway and breathing
	
B9 	 Insert IV line and give fluids
	
B9 	 If intravenous access not possible
	 B10-B12	 Bleeding
	
B10	 Mas

In [16]:
import fitz
from pathlib import Path

pdf_file = Path.cwd() / "data" / "raw" / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

for page_num in range(len(doc)):
    text = doc.load_page(page_num).get_text("text")
    if "Vaginal bleeding" in text:
        print(f"Found on page {page_num + 1}")

doc.close()

Found on page 6
Found on page 22
Found on page 24
Found on page 25
Found on page 26
Found on page 39
Found on page 40
Found on page 162
Found on page 165
Found on page 172
Found on page 176


In [17]:
import fitz
from pathlib import Path

pdf_file = Path.cwd() / "data" / "raw" / "WHO_Essential_Practice_2023.pdf"

doc = fitz.open(pdf_file)

# Page 22 in the PDF = index 21
page = doc.load_page(21)
text = page.get_text("text")

print(text)

doc.close()

QUICK CHECK, RAPID ASSESSMENT AND MANAGEMENT OF WOMEN OF CHILDBEARING AGE
QUICK CHECK
A person responsible for initial reception of women of childbearing age and newborns seeking care should:

 assess the general condition of the careseeker(s) immediately on arrival

 periodically repeat this procedure if the line is long.
If a woman is very sick, talk to her companion.
ASK, CHECK RECORD
LOOK, LISTEN, FEEL
SIGNS
CLASSIFY
TREAT

 Why did you come?
 
→for yourself?
 
→for the baby?

 How old is the baby?

 What is the concern?
Is the woman being wheeled or 
carried in or:

 bleeding vaginally

 convulsing

 looking very ill

 unconscious

 in severe pain

 in labour

 delivery is imminent

 If the woman is or has:

 unconscious (does not answer)

 convulsing

 bleeding

 severe abdominal pain or looks very ill

 headache and visual disturbance

 severe difficulty breathing

 fever

 severe vomiting.
EMERGENCY 
FOR WOMAN

 Transfer woman to a tr

In [18]:
from pathlib import Path

processed_dir = Path.cwd() / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Processed folder created:", processed_dir)

Processed folder created: C:\Users\Helen\Documents\mamacare-ai-danger-sign-assistant\data\processed


In [19]:
import fitz
from pathlib import Path

raw_dir = Path.cwd() / "data" / "raw"

documents = {}

for pdf_file in raw_dir.glob("*.pdf"):
    doc = fitz.open(pdf_file)
    pages = []

    for page_num, page in enumerate(doc):
        text = page.get_text("text")
        if text.strip():  # keep only pages with text
            pages.append({
                "page": page_num + 1,
                "text": text
            })

    documents[pdf_file.name] = pages
    doc.close()

print("Documents processed:")
for name, pages in documents.items():
    print(name, "-", len(pages), "text pages")

Documents processed:
Nigeria_MPCDSR_2022.pdf - 142 text pages
WHO_Antenatal_Care_2016.pdf - 168 text pages
WHO_Essential_Practice_2023.pdf - 178 text pages


In [20]:
def chunk_text(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# Test on one page
sample_text = documents[list(documents.keys())[0]][0]["text"]
sample_chunks = chunk_text(sample_text)

print("Sample chunks created:", len(sample_chunks))
print(sample_chunks[0][:500])

Sample chunks created: 4
 
 
 
FOREWORD 
Reporting and tracking maternal, perinatal and child deaths and  government initiatives to 
reduce preventable deaths remain a major challenge in Nigeria. The first 28 days of life – 
the neonatal period is a critical period for survival of the child. Every day in Nigeria, over 
700 babies die, the highest number of new-born deaths in Africa, and second highest in the 
world. Over 40,000 Nigerian women die each year during child birth. For every maternal 
death, at least seven ne


In [21]:
import pandas as pd

records = []

for document_name, pages in documents.items():

    if "WHO_Antenatal_Care_2016" in document_name:
        source = "WHO"
        year = 2016

    elif "WHO_Essential_Practice_2023" in document_name:
        source = "WHO"
        year = 2023

    elif "MPCDSR_2022" in document_name:
        source = "Federal Ministry of Health, Nigeria"
        year = 2022

    else:
        source = "Unknown"
        year = None

    for page in pages:
        chunks = chunk_text(page["text"])

        for i, chunk in enumerate(chunks):
            records.append({
                "Document_ID": document_name,
                "Source": source,
                "Publication_Year": year,
                "Page_Number": page["page"],
                "Chunk_ID": f"{document_name}_{page['page']}_{i+1}",
                "Language": "English",
                "Chunk_Text": chunk.strip()
            })

knowledge_base = pd.DataFrame(records)

print("Knowledge base created successfully")
print("Total chunks:", len(knowledge_base))

knowledge_base.head()

Knowledge base created successfully
Total chunks: 1756


,Document_ID,Source,Publication_Year,Page_Number,Chunk_ID,Language,Chunk_Text
0,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_1,English,"FOREWORD \nReporting and tracking maternal, pe..."
1,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_2,English,n and strengthening of \nthe health system blo...
2,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_3,English,care providers in providing quality mater...
3,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_4,English,ral Republic of Nigeria
4,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,3,Nigeria_MPCDSR_2022.pdf_3_1,English,ACKNOWLEDGEMENT \nThe Federal Ministry of Heal...


### Knowledge-Base Dataset Construction

The maternal-health documents were converted into a structured dataset by dividing each readable page into overlapping text chunks. Metadata including the document source, publication year, page number, language, and a unique chunk identifier was attached to every text section.

The resulting dataset forms the initial **RAG knowledge base** for the MamaCare AI assistant. Each row represents a searchable passage that can be retrieved when a user asks about a maternal danger sign such as vaginal bleeding, severe abdominal pain, convulsions, severe headache, blurred vision, fever, swelling, or difficulty breathing.

This approach improves retrieval accuracy, supports evidence-based responses, and allows every answer to be traced back to the original WHO or Nigerian guideline.


In [22]:
from pathlib import Path

output_file = Path.cwd() / "data" / "processed" / "knowledge_base_chunks.csv"

knowledge_base.to_csv(output_file, index=False)

print("Knowledge base saved successfully!")
print(output_file)
print("Total chunks:", len(knowledge_base))

Knowledge base saved successfully!
C:\Users\Helen\Documents\mamacare-ai-danger-sign-assistant\data\processed\knowledge_base_chunks.csv
Total chunks: 1756


## knowledge-base dataset Summary

A structured knowledge-base dataset was successfully created from the WHO and Nigerian maternal-health documents. The documents were converted into machine-readable text, divided into overlapping chunks, and enriched with metadata including document source, publication year, page number, language, and unique chunk identifiers.

The final dataset was saved as **knowledge_base_chunks.csv** in the **data/processed** directory and contains **1,756 searchable text chunks**. This dataset forms the initial Retrieval-Augmented Generation (RAG) knowledge base that will be used by the MamaCare AI assistant to retrieve evidence-based maternal danger-sign information and referral guidance from official WHO and Nigerian guidelines.


In [23]:
import pandas as pd
from pathlib import Path

kb_file = Path.cwd() / "data" / "processed" / "knowledge_base_chunks.csv"

knowledge_base = pd.read_csv(kb_file)

print("Knowledge base loaded successfully!")
print("Total records:", len(knowledge_base))

knowledge_base.head()

Knowledge base loaded successfully!
Total records: 1756


,Document_ID,Source,Publication_Year,Page_Number,Chunk_ID,Language,Chunk_Text
0,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_1,English,"FOREWORD \nReporting and tracking maternal, pe..."
1,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_2,English,n and strengthening of \nthe health system blo...
2,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_3,English,care providers in providing quality mater...
3,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_4,English,ral Republic of Nigeria
4,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,3,Nigeria_MPCDSR_2022.pdf_3_1,English,ACKNOWLEDGEMENT \nThe Federal Ministry of Heal...


In [25]:
def search_knowledge_base(query, top_n=5):
    query = query.lower()

    matches = knowledge_base[
        knowledge_base["Chunk_Text"].str.lower().str.contains(query, na=False)
    ]

    # Prefer WHO documents first
    matches = matches.sort_values(by=["Source", "Publication_Year"], ascending=[True, False])

    return matches[["Document_ID", "Source", "Page_Number", "Chunk_Text"]].head(top_n)

In [26]:
results = search_knowledge_base("vaginal bleeding")

for i, row in results.iterrows():
    print("Source:", row["Document_ID"])
    print("Page:", row["Page_Number"])
    print(row["Chunk_Text"][:700])
    print("-" * 80)

Source: WHO_Essential_Practice_2023.pdf
Page: 6
TABLE OF CONTENTS
	
INTRODUCTION
i   Introduction
i2   How to read the Guide
i3   Structure and presentation
i4   Assumptions underlying the guide
A PRINCIPLES OF GOOD CARE
A2   Communication
A3   Workplace and administrative procedures
A4   Standard precautions and cleanliness
A5   Organising a visit
B QUICK CHECK, RAPID ASSESSMENT AND MANAGEMENT OF WOMEN OF CHILDBEARING AGE
B2   Quick check
B3-B7	  Rapid assessment and management
	
B3 	 Airway and breathing
	
B3 	 Circulation (shock)
	
B4-B5	 Vaginal bleeding
	
B6 	 Convulsions or unconscious
	
B6 	 Severe abdominal pain
	
B6 	 Dangerous fever
	
B7 	 Labour
	
B7 	 Other danger signs or symptoms
	
B7 	 If no emergency or priority signs, non
--------------------------------------------------------------------------------
Source: WHO_Essential_Practice_2023.pdf
Page: 7
(Shoulder dystocia)
	
D18	 If multiple births
D19  Care of the mother and newborn within first hour of delivery of placent

In [27]:
def search_knowledge_base(query, top_n=5):
    query = query.lower()

    matches = knowledge_base[
        knowledge_base["Chunk_Text"].str.lower().str.contains(query, na=False)
    ].copy()

    # Remove table of contents and introductory chunks
    matches = matches[
        ~matches["Chunk_Text"].str.lower().str.contains(
            "table of contents|introduction|foreword|acknowledgement",
            na=False
        )
    ]

    # Prioritize WHO Essential Practice because it contains the clinical guidance
    matches["priority"] = matches["Document_ID"].apply(
        lambda x: 0 if "WHO_Essential_Practice_2023" in x else 1
    )

    matches = matches.sort_values(
        by=["priority", "Page_Number"]
    )

    return matches[["Document_ID", "Source", "Page_Number", "Chunk_Text"]].head(top_n)

In [28]:
results = search_knowledge_base("vaginal bleeding")

for _, row in results.iterrows():
    print("Source:", row["Document_ID"])
    print("Page:", row["Page_Number"])
    print(row["Chunk_Text"][:800])
    print("-" * 80)

Source: WHO_Essential_Practice_2023.pdf
Page: 8
D CHILDBIRTH – LABOUR, DELIVERY AND IMMEDIATE POSTPARTUM CARE (CONTINUED)
D26  Advise on postpartum care
	
D26	 Advise on postpartum care and hygiene
	
D26	 Counsel on nutrition
D27  Counsel on birth spacing and family planning
	
D27	 Counsel on the importance of family planning
	
D27	 Lactation amenorrhea method (LAM)
D28  Advise on when to return
	
D28	 Routine postpartum visits
	
D28	 Follow-up visits for problems
	
D28	 Advise on danger signs
	
D28	 Discuss how to prepare for an emergency in postpartum
D29  Home delivery by skilled attendant
	
D29	 Preparation for home delivery
	
D29	 Delivery care
	
D29	 Immediate postpartum care of mother
	
D29	 Postpartum care of newborn
E POSTPARTUM CARE
E2   Postpartum examination of the mother (up to 6 weeks)
E3-E10  Respond to observed signs or 
--------------------------------------------------------------------------------
Source: WHO_Essential_Practice_2023.pdf
Page: 22
rapidly B9 .

 If n

In [30]:
def mamacare_response(user_input):
    user_input = user_input.lower()

    # Simple danger-sign mapping
    if "bleeding" in user_input or "blood" in user_input:
        query = "vaginal bleeding"
    elif "headache" in user_input:
        query = "severe headache"
    elif "abdominal pain" in user_input or "stomach pain" in user_input:
        query = "severe abdominal pain"
    elif "fever" in user_input:
        query = "dangerous fever"
    elif "convulsion" in user_input or "seizure" in user_input:
        query = "convulsions"
    elif "blurred vision" in user_input or "cannot see" in user_input:
        query = "blurred vision"
    elif "breathing" in user_input:
        query = "breathing difficulty"
    else:
        query = user_input

    results = search_knowledge_base(query, top_n=1)

    if results.empty:
        return (
            "I could not find enough information in the approved maternal-health "
            "guidelines. Please consult a qualified healthcare professional or visit "
            "the nearest health facility."
        )

    row = results.iloc[0]

    response = (
        f"Based on WHO maternal-health guidance ({row['Document_ID']}, page {row['Page_Number']}), "
        f"{row['Chunk_Text'][:500]}... "
        "This may represent a pregnancy danger sign and should be assessed by a qualified "
        "healthcare professional as soon as possible."
    )

    return response

In [31]:
print(mamacare_response("I am bleeding during pregnancy"))

Based on WHO maternal-health guidance (WHO_Essential_Practice_2023.pdf, page 8), D CHILDBIRTH – LABOUR, DELIVERY AND IMMEDIATE POSTPARTUM CARE (CONTINUED)
D26  Advise on postpartum care
	
D26	 Advise on postpartum care and hygiene
	
D26	 Counsel on nutrition
D27  Counsel on birth spacing and family planning
	
D27	 Counsel on the importance of family planning
	
D27	 Lactation amenorrhea method (LAM)
D28  Advise on when to return
	
D28	 Routine postpartum visits
	
D28	 Follow-up visits for problems
	
D28	 Advise on danger signs
	
D28	 Discuss how to prepare for an emergency in... This may represent a pregnancy danger sign and should be assessed by a qualified healthcare professional as soon as possible.


In [32]:
def search_knowledge_base(query, top_n=5):
    query = query.lower()

    matches = knowledge_base[
        knowledge_base["Chunk_Text"].str.lower().str.contains(query, na=False)
    ].copy()

    # Remove non-clinical navigation pages
    matches = matches[
        ~matches["Chunk_Text"].str.lower().str.contains(
            "table of contents|introduction|foreword|acknowledgement|advise on postpartum care|birth spacing",
            na=False
        )
    ]

    # Give higher priority to chunks containing emergency language
    def score(text):
        text = text.lower()
        score = 0
        if "vaginal bleeding" in text:
            score += 5
        if "urgent" in text:
            score += 4
        if "dangerous" in text:
            score += 3
        if "refer woman urgently to hospital" in text:
            score += 5
        if "heavy bleeding" in text:
            score += 4
        return score

    matches["score"] = matches["Chunk_Text"].apply(score)

    matches = matches.sort_values(by=["score", "Page_Number"], ascending=[False, True])

    return matches[["Document_ID", "Source", "Page_Number", "Chunk_Text"]].head(top_n)

In [33]:
print(mamacare_response("I am bleeding during pregnancy"))

Based on WHO maternal-health guidance (WHO_Essential_Practice_2023.pdf, page 22), M/IV if bleeding continues.

 If suspect possible complicated abortion, give appropriate IM/IV antibiotics B15.

 Refer woman urgently to hospital B17.
This may be abortion, 
menorrhagia, 
ectopic pregnancy.
LIGHT BLEEDING

 Examine woman as on B19.

 If pregnancy not likely, refer to other clinical guidelines.
LATE PREGNANCY
(uterus above umbilicus)
ANY BLEEDING IS DANGEROUS
DO NOT do vaginal examination, but:

 Insert an IV line B9 .

 Give fluids rapidly if heavy bleeding or shock... This may represent a pregnancy danger sign and should be assessed by a qualified healthcare professional as soon as possible.


Creating Bilingual response generation Template
For example:

User: “I am bleeding during pregnancy.”

English response:

Bleeding during pregnancy can be a serious danger sign. According to WHO guidance, any vaginal bleeding in late pregnancy should be treated as an emergency. Please go to the nearest hospital or health facility immediately.

Pidgin response:

If blood dey come out from your body during pregnancy, e fit be danger sign. According to WHO guidance, any bleeding for late pregnancy na emergency. Abeg go the nearest hospital or health centre immediately make health worker check you.

In [34]:
bilingual_templates = {
    "vaginal bleeding": {
        "english": (
            "Bleeding during pregnancy can be a serious danger sign. "
            "According to WHO guidance, any vaginal bleeding in late pregnancy should be treated as an emergency. "
            "Please go to the nearest hospital or health facility immediately."
        ),
        "pidgin": (
            "If blood dey come out from your body during pregnancy, e fit be danger sign. "
            "According to WHO guidance, any bleeding for late pregnancy na emergency. "
            "Abeg go the nearest hospital or health centre immediately make health worker check you."
        )
    },

    "severe headache": {
        "english": (
            "A severe headache during pregnancy can be a danger sign, especially if it is persistent or associated with blurred vision. "
            "Please seek medical assessment as soon as possible."
        ),
        "pidgin": (
            "If strong headache no gree stop during pregnancy, e fit be danger sign, especially if your eye dey blur. "
            "Abeg go hospital make health worker check you quickly."
        )
    },

    "dangerous fever": {
        "english": (
            "Fever during pregnancy may be a sign of infection. "
            "Please seek medical care as soon as possible."
        ),
        "pidgin": (
            "If your body hot well-well during pregnancy, e fit mean infection. "
            "Abeg go hospital quickly make health worker check you."
        )
    }
}

print("Bilingual templates created successfully.")

Bilingual templates created successfully.


In [35]:
def mamacare_bilingual(user_input, language="english"):
    text = user_input.lower()

    if "bleeding" in text or "blood" in text:
        key = "vaginal bleeding"
    elif "headache" in text:
        key = "severe headache"
    elif "fever" in text or "hot" in text:
        key = "dangerous fever"
    else:
        return {
            "english": (
                "I do not have enough reliable information to answer this safely. "
                "Please consult a qualified healthcare professional."
            ),
            "pidgin": (
                "I no get enough correct information to answer this safely. "
                "Abeg talk to qualified health worker or go hospital."
            )
        }.get(language, "Language not supported.")

    return bilingual_templates[key][language]

In [36]:
print(mamacare_bilingual("I am bleeding during pregnancy", "english"))

Bleeding during pregnancy can be a serious danger sign. According to WHO guidance, any vaginal bleeding in late pregnancy should be treated as an emergency. Please go to the nearest hospital or health facility immediately.


In [37]:
print(mamacare_bilingual("Blood dey come out during pregnancy", "pidgin"))

If blood dey come out from your body during pregnancy, e fit be danger sign. According to WHO guidance, any bleeding for late pregnancy na emergency. Abeg go the nearest hospital or health centre immediately make health worker check you.


In [38]:
def mamacare_followup_bleeding():
    return {
        "english": (
            "I’m sorry you’re experiencing bleeding during pregnancy. "
            "Can you tell me if the bleeding is **light spotting** or **heavy bleeding**?"
        ),
        "pidgin": (
            "I sorry say this dey happen. Blood during pregnancy fit be danger sign. "
            "Abeg tell me: the blood dey come **small-small** or **e dey come plenty**?"
        )
    }

In [39]:
print(mamacare_followup_bleeding()["pidgin"])

I sorry say this dey happen. Blood during pregnancy fit be danger sign. Abeg tell me: the blood dey come **small-small** or **e dey come plenty**?


In [40]:
priority_followups = {
    "vaginal bleeding": {
        "english": (
            "I’m sorry you’re experiencing bleeding during pregnancy. "
            "Can you tell me if the bleeding is **light spotting** or **heavy bleeding**?"
        ),
        "pidgin": (
            "I sorry say this dey happen. Blood during pregnancy fit be danger sign. "
            "Abeg tell me: the blood dey come **small-small** or **e dey come plenty**?"
        )
    },

    "severe headache": {
        "english": (
            "I’m sorry you’re having a severe headache. "
            "Are you also having **blurred vision, swelling of the face or hands, or severe abdominal pain**?"
        ),
        "pidgin": (
            "I sorry say your head dey pain you well-well. "
            "Your eye dey blur, your face or hand dey swell, or you get serious belle pain too?"
        )
    },

    "reduced fetal movement": {
        "english": (
            "I’m sorry you’re worried about your baby’s movement. "
            "Have you noticed that the baby is **moving much less than usual** or **has stopped moving completely**?"
        ),
        "pidgin": (
            "I understand say you dey worry about your baby movement. "
            "The baby dey move **less than before** or **e don stop to move completely**?"
        )
    }
}

print("Priority follow-up questions created.")

Priority follow-up questions created.


In [41]:
priority_followups = {
    "vaginal bleeding": {
        "english": (
            "I’m sorry you’re experiencing bleeding during pregnancy. "
            "Can you tell me if the bleeding is **light spotting** or **heavy bleeding**?"
        ),
        "pidgin": (
            "I sorry say this dey happen. Blood during pregnancy fit be danger sign. "
            "Abeg tell me: the blood dey come **small-small** or **e dey come plenty**?"
        )
    },

    "severe headache": {
        "english": (
            "I’m sorry you’re having a severe headache. "
            "Are you also having **blurred vision, swelling of the face or hands, or severe abdominal pain**?"
        ),
        "pidgin": (
            "I sorry say your head dey pain you well-well. "
            "Your eye dey blur, your face or hand dey swell, or you get serious belle pain too?"
        )
    },

    "reduced fetal movement": {
        "english": (
            "I’m sorry you’re worried about your baby’s movement. "
            "Have you noticed that the baby is **moving much less than usual** or **has stopped moving completely**?"
        ),
        "pidgin": (
            "I understand say you dey worry about your baby movement. "
            "The baby dey move **less than before** or **e don stop to move completely**?"
        )
    }
}

print("Priority follow-up questions created.")

Priority follow-up questions created.


In [42]:
priority_responses = {
    "vaginal bleeding": {
        "english": (
            "According to WHO guidance, **any vaginal bleeding in late pregnancy is a danger sign**. "
            "Heavy bleeding is an emergency. Please go to the nearest hospital or health facility immediately. "
            "Do not insert anything into the vagina and seek urgent medical assessment."
        ),
        "pidgin": (
            "According to WHO guidance, **any blood wey dey come out for late pregnancy na danger sign**. "
            "If the bleeding plenty, na emergency. Abeg go the nearest hospital or health centre immediately. "
            "No put anything for your vagina and make health worker check you quickly."
        )
    },

    "severe headache": {
        "english": (
            "A severe headache during pregnancy, especially with blurred vision or swelling, can be a sign of **pre-eclampsia**, "
            "which is a serious pregnancy complication. Please go to a hospital or health facility immediately for assessment."
        ),
        "pidgin": (
            "Strong headache during pregnancy, especially if your eye dey blur or your body dey swell, fit be sign of **pre-eclampsia**. "
            "This one serious. Abeg go hospital immediately make health worker check you."
        )
    },

    "reduced fetal movement": {
        "english": (
            "Reduced or absent baby movement can be a danger sign. "
            "If your baby is moving much less than usual or has stopped moving, please go to the nearest hospital or health facility immediately for assessment."
        ),
        "pidgin": (
            "If your baby no dey move like before or e don stop to move, e fit be danger sign. "
            "Abeg go the nearest hospital or health centre immediately make health worker check the baby."
        )
    }
}

print("Priority danger-sign responses created.")

Priority danger-sign responses created.


In [43]:
def mamacare_priority_triage(user_input):
    language = detect_language(user_input)
    text = user_input.lower()

    if "bleeding" in text or "blood" in text:
        key = "vaginal bleeding"
    elif "headache" in text or "head dey pain" in text:
        key = "severe headache"
    elif "baby not moving" in text or "baby no dey move" in text or "fetal movement" in text:
        key = "reduced fetal movement"
    else:
        return {
            "language": language,
            "follow_up": None,
            "response": {
                "english": (
                    "I do not have enough reliable information to answer this safely. "
                    "Please consult a qualified healthcare professional."
                ),
                "pidgin": (
                    "I no get enough correct information to answer this safely. "
                    "Abeg talk to qualified health worker or go hospital."
                )
            }[language]
        }

    return {
        "language": language,
        "follow_up": priority_followups[key][language],
        "response": priority_responses[key][language]
    }

In [45]:
def detect_language(text):
    text = text.lower()

    pidgin_words = [
        "dey", "abeg", "waka", "body", "well-well",
        "fit", "hospital", "blood dey", "make", "you"
    ]

    score = sum(word in text for word in pidgin_words)

    if score >= 2:
        return "pidgin"
    return "english"

print("Language detection function loaded.")

Language detection function loaded.


Testing all three danger sign

In [46]:
result = mamacare_priority_triage("Blood dey come out during pregnancy")
print("Language:", result["language"])
print("Follow-up:", result["follow_up"])
print("Response:", result["response"])

Language: pidgin
Follow-up: I sorry say this dey happen. Blood during pregnancy fit be danger sign. Abeg tell me: the blood dey come **small-small** or **e dey come plenty**?
Response: According to WHO guidance, **any blood wey dey come out for late pregnancy na danger sign**. If the bleeding plenty, na emergency. Abeg go the nearest hospital or health centre immediately. No put anything for your vagina and make health worker check you quickly.


In [48]:
def detect_language(text):
    text = text.lower()

    pidgin_words = [
        "dey", "abeg", "waka", "body", "well-well",
        "fit", "hospital", "blood dey", "make", "you"
    ]

    score = sum(word in text for word in pidgin_words)

    if score >= 2:
        return "pidgin"
    return "english"


def mamacare_conversation(symptom, answer, language="english"):
    symptom = symptom.lower()
    answer = answer.lower()

    # ---------------- VAGINAL BLEEDING ----------------
    if "bleeding" in symptom or "blood" in symptom:

        heavy = ["heavy", "a lot", "plenty", "many", "full pad", "e dey come plenty"]
        light = ["light", "spotting", "small", "small-small", "little", "e dey come small"]

        if any(word in answer for word in heavy):
            return {
                "english": (
                    "Heavy bleeding during pregnancy can be an emergency. According to WHO guidance, "
                    "heavy vaginal bleeding may be associated with serious pregnancy complications. "
                    "Please go to the nearest hospital or health facility immediately. "
                    "Do not wait for the bleeding to stop by itself."
                ),
                "pidgin": (
                    "If the blood dey come plenty during pregnancy, e fit be emergency. "
                    "According to WHO guidance, heavy bleeding fit mean serious pregnancy problem. "
                    "Abeg go the nearest hospital immediately. No wait make the bleeding stop by itself."
                )
            }[language]

        elif any(word in answer for word in light):
            return {
                "english": (
                    "Even light bleeding or spotting during pregnancy should be checked by a healthcare professional. "
                    "Please arrange to visit a hospital or health facility as soon as possible today."
                ),
                "pidgin": (
                    "Even if the blood na small-small, make health worker still check you today. "
                    "Abeg go hospital or health centre as soon as possible."
                )
            }[language]

        else:
            return {
                "english": (
                    "Because I cannot determine how much bleeding you are having, please seek medical assessment as soon as possible."
                ),
                "pidgin": (
                    "Because I no fit know how much blood dey come out, abeg go hospital make health worker check you quickly."
                )
            }[language]

    # ---------------- SEVERE HEADACHE ----------------
    if "headache" in symptom or "head dey pain" in symptom:

        severe = ["yes", "blurred", "vision", "swelling", "swell", "eye dey blur", "hand dey swell"]

        if any(word in answer for word in severe):
            return {
                "english": (
                    "A severe headache together with blurred vision or swelling can be a sign of pre-eclampsia, "
                    "which is a serious pregnancy complication. Please go to a hospital immediately for assessment."
                ),
                "pidgin": (
                    "Strong headache plus blurred vision or swelling fit be sign of pre-eclampsia. "
                    "This one serious. Abeg go hospital immediately make health worker check you."
                )
            }[language]

        else:
            return {
                "english": (
                    "Because you are pregnant and have a severe headache, you should still be assessed by a healthcare professional today."
                ),
                "pidgin": (
                    "Because you get strong headache during pregnancy, make health worker still check you today."
                )
            }[language]

    # ---------------- REDUCED FETAL MOVEMENT ----------------
    if "baby" in symptom or "fetal movement" in symptom:

        stopped = ["stopped", "no movement", "not moving", "completely", "don stop"]

        if any(word in answer for word in stopped):
            return {
                "english": (
                    "If your baby has stopped moving completely, this can be a danger sign. "
                    "Please go to the nearest hospital or health facility immediately for assessment."
                ),
                "pidgin": (
                    "If your baby don stop to move completely, e fit be danger sign. "
                    "Abeg go the nearest hospital immediately make health worker check the baby."
                )
            }[language]

        else:
            return {
                "english": (
                    "If your baby is moving much less than usual, you should be assessed by a healthcare professional today."
                ),
                "pidgin": (
                    "If your baby dey move less than before, make health worker check you today."
                )
            }[language]

    return {
        "english": "Please seek assessment from a qualified healthcare professional.",
        "pidgin": "Abeg go hospital make qualified health worker check you."
    }[language]


print("MamaCare conversation engine loaded successfully.")

MamaCare conversation engine loaded successfully.


In [49]:
print(mamacare_conversation(
    symptom="I have severe headache during pregnancy",
    answer="Yes, my eyes are blurred and my hands are swollen",
    language="english"
))

A severe headache together with blurred vision or swelling can be a sign of pre-eclampsia, which is a serious pregnancy complication. Please go to a hospital immediately for assessment.


In [50]:
print(mamacare_conversation(
    symptom="My baby is not moving today",
    answer="The baby has stopped moving completely",
    language="english"
))

If your baby has stopped moving completely, this can be a danger sign. Please go to the nearest hospital or health facility immediately for assessment.


In [51]:
print(mamacare_conversation(
    symptom="My head dey pain me well-well for pregnancy",
    answer="Yes, my eye dey blur and my hand dey swell",
    language="pidgin"
))

Strong headache plus blurred vision or swelling fit be sign of pre-eclampsia. This one serious. Abeg go hospital immediately make health worker check you.


In [52]:
print(mamacare_conversation(
    symptom="My baby no dey move today",
    answer="The baby don stop to move completely",
    language="pidgin"
))

If your baby don stop to move completely, e fit be danger sign. Abeg go the nearest hospital immediately make health worker check the baby.
